In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

In [ ]:
df = pd.read_parquet('sciezka.parquet') 
print(df.head(5))

In [ ]:
#pomiary nieregularne, więc spróbujmy je jakoś uporządkować i na początek zrobić agregację do 5minut, można potem jakieś nie wiem po sekundach jakieś minuty
#czasami prościej jest na przykład taka 1minuta, bo może różnice będą lepiej widoczne

df_5min = df["przeplyw"].resample("5min").agg(
    ["mean", "min", "max", "std", "count"]
)

df_5min.columns = ["przeplyw_mean",
    "przeplyw_min",
    "przeplyw_max",
    "przeplyw_std",
    "liczba_pomiarow"
]

print(df_5min.head())

In [ ]:
df_5min["rolling_mean_1h"] = (
    df_5min["przeplyw_mean"].rolling(window=12, min_periods=6).mean()
)

df_5min["rolling_min_1h"] = (
    df_5min["przeplyw_min"].rolling(window=12, min_periods=6).min()
)

df_5min["rolling_max_1h"] = (
    df_5min["przeplyw_max"].rolling(window=12, min_periods=6).max()
)

df_5min["rolling_std_1h"] = (
    df_5min["przeplyw_std"].rolling(window=12, min_periods=6).std()
)

df_5min["diff"] = df_5min["przeplyw_mean"].diff()
df_5min["ewm"] = (
    df_5min["przeplyw_mean"].ewm(span=12, adjust=False).mean()
)

In [ ]:
#podział danych chronologicznie
#noo narazie to tak na oko, czasami może być właśnie tak, że ten podział 70:30 nie bedzie skuteczny
#najlepiej nauczyć model na takim zdrowym okresie

split = int(len(df_5min)*0.7)

train = df_5min.iloc[:split].copy()
test = df_5min.iloc[split:].copy()

print(f"Train: {train.index.min()}, {train.index.max()}")
print(f"Test: {test.index.min()}, {test.index.max()}")

In [ ]:
#MAD - mediana bezwzględnych odchyleń od mediany 
def mad(x):
    return np.median(np.abs(x-np.median(x)))

In [ ]:
#rolling MAD - wykrywanie awarii 
#spróbujmy 24h, ale możemy to zmieniać
#na przykład też można ostatnie 30dni też spróbować
#możemy spróbować również z innymi, ale narazie pokombinujmy z mean


#nie chcemy aktualnej wartości, bo bazujemy na danych historycznych
signal = df_5min['przeplyw_mean'].shift(1)
#window = 288, bo mamy co 5min pomiary, a 24h ma 1440min
window = 288
k = 4

median = signal.rolling(window=window, min_periods=120).median()
rolling_mad = signal.rolling(window=window, min_periods=100).apply(mad, raw = True)

df_5min["mad_threshold"] = median + k* 1.4826 * rolling_mad
df_5min["anomaly_mad"] = (
    df_5min["przeplyw_min"] > df_5min["mad_threshold"]
)

In [ ]:
#wizualizacja jakaś

plt.figure(figsize=(6,8))
plt.plot(df_5min.index, df_5min["przeplyw_mean"], label="Przeplyw")
plt.plot(df_5min.index, df_5min["mad_threshold"], label="RollingMAD", color="orange")

#na czerwono niech będą wykryte anomalie 
anomalies = df_5min[df_5min["anomaly_mad"]]
plt.scatter(anomalies.index, anomalies["przeplyw_mean"], color="red", label="Anomalie")
plt.show()

### IsolationForest
Algorytm uczenia nienadzorowanego, który służy do wykrywania anomalii, nietypowych zachowań (wykorzystuje losowe drzewa do izolowania nietypowych obserwacji -> umożliwia identyfikację nieprawidłowości z danych i wychodzi z założenia, że nietypowe obserwacje jest zazwyczaj łatwiej odizolować od pozostałych danych)

In [ ]:
pip install isotree

In [ ]:
#isotree - ponieważ jest to bibliotrka z cpp, która zawiera ten wariant, a ten wariant jest odporny na ewentualne nany
from isotree import IsolationForest 
from sklearn.preprocessing import StandardScaler

In [ ]:
X = df_5min.copy()
X = X.dropna(axis=1, how="all")

X_train = X.iloc[:split]
X_test = X.iloc[split:]

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

iso = IsolationForest(ntrees=200, sample_size=256, ndim=3, missing_action="impute", random_seed=42)

iso.fit(X_train_scaled)

In [ ]:
iso_scores_train = pd.Series(iso.predict(X_train_scaled,output="score"), index= X_train.index)
iso_scores_test = pd.Series(iso.predict(X_test_scaled,output="score"), index= X_test.index)


# wyznaczanie progu - narazie spróbujmy z kwantylem, ale są różne metody
#np. adapcyjne progi takie jak tukey, iqr, mad (rolling najlepiej)

threshold = iso_scores_train.quantile(0.99)
anomalies = iso_scores_test > threshold
print("Prog", threshold)
print("Liczba anomalii", anomalies.sum())